# Day 1 · 02 · SQL Programming & Analytics Architecture

![Architecture patterns to Power BI Gold](../../../assets/images/architecture_patterns_to_powerbi.png)

How this visual helps: it links architecture vocabulary to the Power BI outcome. The point is not to memorize patterns, but to decide which pattern shapes the Gold contract.

This demo turns source discovery into professional BI SQL habits. We use small, disposable examples to practice CTEs, windows, variables, SQL objects and architecture vocabulary. The full Kimball Gold model is intentionally left for Workshop 1.

## Business Scenario

RetailHub analysts can now find the source tables and inspect Delta metadata. The next risk is SQL quality: repeated filters, nested queries nobody can review, wrong distinct counts, unclear ownership of KPI logic and architecture terms used without precision.

Today we build the mental and SQL patterns required before Workshop 1. The output is not a final Gold model; it is a set of repeatable habits for writing, reviewing and operationalizing BI SQL.

## Power BI Developer Lens

This notebook teaches SQL patterns because Power BI semantic models depend on visible grain, filters and KPI definitions. The goal is not to become a data engineer; the goal is to read and write BI SQL that a data platform team can review and that a report owner can trust.

| SQL practice | Power BI impact |
|---|---|
| CTEs with named steps | reviewers can see filters, grain and business logic |
| explicit grain checks | relationships and DAX measures do not double-count |
| NULL-safe calculations | KPI cards do not fail or show misleading blanks |
| window functions | trend and ranking logic can be validated upstream |
| pre-aggregation | report pages scan less data and cost less |
| metric ownership | shared definitions stay in Gold; visual-specific logic stays in DAX |

Expected observation: good Databricks SQL makes the future Power BI model easier, not more complicated.

## Learning Objectives

By the end of this notebook you can:

- structure BI SQL with CTEs and multi-CTEs,
- use `RANK`, `ROW_NUMBER`, `LAG` and running totals,
- calculate NULL-safe ratios,
- avoid distinct-count traps,
- choose between temporary views, views, materialized views, streaming tables and tables,
- use SQL parameters and session variables for repeatable analysis,
- recognize SQL scripting as an automation pattern for Databricks SQL,
- explain Medallion, Kimball, Inmon, Data Vault and Data Mesh,
- explain SCD Type 1 vs Type 2.

Documentation starting points: [Databricks SQL language reference](https://docs.databricks.com/sql/language-manual/), [DECLARE VARIABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-declare-variable), [SET VARIABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-aux-set-variable), [SQL scripting](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-scripting), [CREATE VIEW](https://docs.databricks.com/sql/language-manual/sql-ref-syntax-ddl-create-view.html), [materialized views](https://docs.databricks.com/ldp/dbsql/materialized.html).

## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `silver.order_lines`.

In [ ]:
%run ../../../setup/00_setup

### Configuration

Confirm the active catalog and schemas. This is intentionally Python because it reads variables produced by the shared setup notebook.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

This notebook requires the four Silver source tables created by `data/generate_training_dataset.ipynb`. The check fails early, before SQL examples start returning confusing errors.

In [ ]:
required_tables = [
    f"{SILVER}.customers",
    f"{SILVER}.products",
    f"{SILVER}.sales_orders",
    f"{SILVER}.order_lines",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run data/generate_training_dataset.ipynb first.")
print("[OK] Silver source tables are available.")

## 1. SQL for BI: Reviewable, Explicit, Testable

![BI SQL Quality Patterns](../../../assets/images/day1_02_sql_quality_patterns.png)

How this visual helps: it makes SQL quality reviewable. Participants should look for grain, filters, null handling and performance signals before trusting a query as a BI source.

Good analytics SQL exposes grain, filters and business logic. The goal is a query another BI developer can review and test, not a clever one-liner.

Hands-on step: run the next cells and identify where each query states its grain, filter and aggregation rule.

Expected observation: the result is easier to review when every step has a business name.

### SQL Pattern Documentation Anchors

Use these references for the SQL features used in the first half of this notebook:

| Feature | Working definition | Official reference |
|---|---|---|
| `SELECT` | SQL statement that composes a result set from tables, views, CTEs and expressions | [SELECT](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select) |
| CTE | named temporary result set scoped to one SQL statement | [Common table expression](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-select-cte) |
| Built-in function | Databricks SQL function for scalar, aggregate, window, date, string and complex-type logic | [Built-in functions](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-functions-builtin) |
| `CASE` | conditional expression for business flags and controlled labels | [CASE expression](https://docs.databricks.com/aws/en/sql/language-manual/functions/case) |
| `COUNT(DISTINCT ...)` | aggregate pattern for counting unique entities, not physical rows | [Aggregate functions](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-functions-builtin) |

Trainer note: link to the language reference when syntax is the issue; use the local examples when the issue is BI grain or KPI meaning.


### BI SQL Review Checklist

Use this checklist before a query becomes a shared BI source.

| Review item | Definition | Common failure |
|---|---|---|
| Grain | what one row represents | mixing order rows and line rows |
| Filters | business scope of the result | hidden status/date filters |
| Joins | how tables are related | duplicate facts after many-to-many joins |
| Null handling | how missing values behave | divide-by-zero or blank KPI cards |
| Distinct count | what entity is counted | counting lines as orders |
| Naming | whether fields read like business terms | leaking source-system names into BI |
| Performance smell | query pattern likely to over-scan | broad `SELECT *`, late filters, repeated nested logic |

Expected observation: reviewable SQL is not shorter SQL. It is SQL where grain, filters and business logic are visible.


### Prove Source Grain

Before writing metrics, prove what one row means. A line-grain table and an order-grain table answer different BI questions.

In [ ]:
%sql
SELECT
  'sales_orders' AS object_name,
  'one row per order' AS expected_grain,
  COUNT(*) AS rows,
  COUNT(DISTINCT order_id) AS distinct_business_keys
FROM silver.sales_orders
UNION ALL
SELECT
  'order_lines',
  'one row per order line',
  COUNT(*),
  COUNT(DISTINCT line_id)
FROM silver.order_lines

### CTE Definitions

![CTE Steps for BI SQL](../../../assets/images/day1_02_cte_steps_for_bi_sql_ai.png)

How this visual helps: it shows CTEs as named review steps, not just SQL syntax. The training point is to isolate scope, prove grain, aggregate metrics and produce a result another BI developer can review.

A CTE, or common table expression, is a named query block inside one SQL statement. Use it to name a transformation step, isolate filters and make a metric query reviewable.

A multi-CTE query chains several named blocks. The pattern is useful when each block has one job: filter source rows, aggregate to a clear grain, rank or calculate the final output.

Pitfall: a CTE is not automatically a persisted table. It improves readability, but the optimizer still decides the execution plan.


### CTE: Name One Transformation Step

A CTE makes a filter or transformation visible before aggregation. Use it when the named step has business meaning.

In [ ]:
%sql
WITH completed_lines AS (
  SELECT
    order_id,
    product_id,
    quantity,
    unit_price,
    line_revenue
  FROM silver.order_lines
  WHERE status = 'Completed'
)
SELECT
  product_id,
  ROUND(SUM(line_revenue), 2) AS revenue
FROM completed_lines
GROUP BY product_id
ORDER BY revenue DESC
LIMIT 10

### Multi-CTE: Filter, Aggregate, Rank

Use one named CTE per step when logic has a sequence. This pattern becomes important in Workshop 1 when raw business events become dimensions, facts and dashboard tables.

In [ ]:
%sql
WITH completed AS (
  SELECT order_id, customer_id, line_revenue
  FROM silver.order_lines
  WHERE status = 'Completed'
),
by_customer AS (
  SELECT customer_id, ROUND(SUM(line_revenue), 2) AS revenue
  FROM completed
  GROUP BY customer_id
),
ranked AS (
  SELECT customer_id, revenue, DENSE_RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
  FROM by_customer
)
SELECT *
FROM ranked
WHERE revenue_rank <= 10
ORDER BY revenue_rank

## 2. Metric Traps

Shared KPI logic belongs upstream when multiple reports depend on the same definition. Power BI can still own visual-specific measures, but Gold should protect grain, shared filters and approved definitions.

### Lines Are Not Orders

The easiest BI mistake is to count line rows as if they were order rows. This query makes the overcount visible.

In [ ]:
%sql
SELECT
  COUNT(*) AS line_count,
  COUNT(DISTINCT order_id) AS order_count,
  COUNT(*) - COUNT(DISTINCT order_id) AS overcount_if_lines_are_orders
FROM silver.order_lines
WHERE status = 'Completed'

### NULL-Safe Function Definitions

`NULLIF(a, b)` returns `NULL` when `a = b`; otherwise it returns `a`. In BI ratios, `NULLIF(denominator, 0)` protects dashboards from divide-by-zero errors.

`COALESCE(a, b, c)` returns the first non-null value. Use it to make missing attributes visible as `Unknown`, not as silent blanks.

`COUNT(DISTINCT ...)` counts unique values. Use it for entities such as orders or customers, but always confirm the grain first because distinct counts can hide duplicated joins.

Documentation: [NULL semantics](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-null-semantics), [`nullif`](https://docs.databricks.com/aws/en/sql/language-manual/functions/nullif), [`coalesce`](https://docs.databricks.com/aws/en/sql/language-manual/functions/coalesce), [built-in functions](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-functions-builtin).


### NULL-Safe Calculations

Ratios should be calculated after aggregation, and the denominator must be protected with `NULLIF`. This avoids divide-by-zero failures and avoids averaging row-level percentages incorrectly.

In [ ]:
%sql
SELECT
  channel,
  ROUND(SUM(line_revenue), 2) AS revenue,
  ROUND(SUM(line_margin), 2) AS gross_margin,
  ROUND(100.0 * SUM(line_margin) / NULLIF(SUM(line_revenue), 0), 2) AS margin_rate_pct
FROM silver.order_lines
WHERE status = 'Completed'
GROUP BY channel
ORDER BY revenue DESC

## 3. Window Functions for BI Patterns

Window functions add analytical context without collapsing detail rows. Use them when the business question says: rank, sequence, previous value, cumulative value or duplicate candidate.

### Window Function Definitions

A window function calculates across a set of related rows without collapsing the result like `GROUP BY` does. Use window functions for ranks, deduplication, trends and running totals.

`RANK` assigns the same rank to ties and leaves gaps. `ROW_NUMBER` assigns exactly one sequence number per row, which is useful for choosing one representative row.

`LAG` reads a value from a previous row in the same partition. Running totals use a window frame such as `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`.

Pitfall: window functions require a clear `PARTITION BY` and `ORDER BY`. Without them, the business meaning of the result is unstable.

Documentation: [window functions](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-window-functions), [`rank`](https://docs.databricks.com/aws/en/sql/language-manual/functions/rank), [`row_number`](https://docs.databricks.com/aws/en/sql/language-manual/functions/row_number), [`lag`](https://docs.databricks.com/aws/en/sql/language-manual/functions/lag).


### RANK: Top Products in Each Category

`RANK` keeps ties visible. That matters when a Top N visual should not hide equal performers.

In [ ]:
%sql
WITH product_revenue AS (
  SELECT
    category,
    product_id,
    ROUND(SUM(line_revenue), 2) AS revenue
  FROM silver.order_lines
  WHERE status = 'Completed'
  GROUP BY category, product_id
),
ranked AS (
  SELECT
    category,
    product_id,
    revenue,
    RANK() OVER (PARTITION BY category ORDER BY revenue DESC) AS rank_in_category
  FROM product_revenue
)
SELECT *
FROM ranked
WHERE rank_in_category <= 3
ORDER BY category, rank_in_category

### ROW_NUMBER: Duplicate Candidate Review

`ROW_NUMBER` is useful for deterministic review lists. It does not prove a row is wrong; it identifies rows worth investigating.

In [ ]:
%sql
WITH numbered AS (
  SELECT
    order_id,
    product_id,
    line_id,
    ROW_NUMBER() OVER (PARTITION BY order_id, product_id ORDER BY line_id) AS rn
  FROM silver.order_lines
)
SELECT *
FROM numbered
WHERE rn > 1
ORDER BY order_id, product_id, line_id
LIMIT 20

### LAG: Month-over-Month Trend

`LAG` compares the current row with a previous row in the ordered analytical window.

In [ ]:
%sql
WITH monthly AS (
  SELECT
    DATE_FORMAT(order_date, 'yyyy-MM') AS year_month,
    ROUND(SUM(line_revenue), 2) AS revenue
  FROM silver.order_lines
  WHERE status = 'Completed'
  GROUP BY DATE_FORMAT(order_date, 'yyyy-MM')
)
SELECT
  year_month,
  revenue,
  LAG(revenue) OVER (ORDER BY year_month) AS previous_month_revenue,
  ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY year_month))
    / NULLIF(LAG(revenue) OVER (ORDER BY year_month), 0), 2) AS mom_pct
FROM monthly
ORDER BY year_month

### Running Total

A running total is a cumulative window. The `ROWS BETWEEN` frame makes the calculation explicit and reviewable.

In [ ]:
%sql
WITH daily AS (
  SELECT
    order_date,
    ROUND(SUM(line_revenue), 2) AS revenue
  FROM silver.order_lines
  WHERE status = 'Completed'
  GROUP BY order_date
)
SELECT
  order_date,
  revenue,
  ROUND(SUM(revenue) OVER (ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS running_revenue
FROM daily
ORDER BY order_date
LIMIT 30

## 4. SQL Objects for BI Workloads

![SQL Objects for BI](../../../assets/images/day1_02_sql_objects_for_bi.png)

How this visual helps: it positions each SQL object by lifecycle. Temporary objects support exploration; durable tables and governed views support shared BI consumption.

Different SQL objects solve different lifecycle problems. Keep temporary objects for demos and exploration; use governed objects when Power BI or another team depends on the result.

Short reference:

| Object | Persistence | Refresh behavior | Use in this course |
|---|---|---|---|
| CTE | one statement | recomputed inside the statement | readable query blocks |
| TEMP VIEW | session only | recomputed on query | disposable demo intermediate logic |
| VIEW | schema object | recomputed on query | shared governed logic without storage |
| MATERIALIZED VIEW | schema object | refreshed and stored | expensive BI aggregates, when enabled |
| STREAMING TABLE | schema object | managed incremental or streaming refresh | pipeline concepts, not this demo |
| TABLE | schema object | explicit pipeline/job refresh | durable BI contract in Workshop 1 |

Expected observation: Day1_02 uses disposable views and small examples; Workshop 1 creates durable Gold objects.


### SQL Object Documentation Anchors

| Object / statement | Working definition | Official reference |
|---|---|---|
| Table | stored structured dataset; default durable object for Gold contracts | [Tables and views](https://docs.databricks.com/aws/en/data-engineering/tables-views) |
| `CREATE TABLE` | creates managed, external or temporary tables | [CREATE TABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-table-using) |
| View | virtual table defined by a query; no physical result storage | [CREATE VIEW](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-view) |
| Temporary view | session-scoped view, dropped when the session ends | [CREATE VIEW](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-view) |
| Materialized view | precomputed query result refreshed manually or on a schedule | [CREATE MATERIALIZED VIEW](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-materialized-view) |
| Streaming table | Unity Catalog managed table with incremental or streaming processing logic | [CREATE STREAMING TABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-streaming-table) |


### TEMP VIEW for Session Reuse

A temp view lets one notebook session reuse a named result. It is useful for demos and exploration, but it is not a governed BI contract.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW demo_completed_channel AS
SELECT
  channel,
  COUNT(DISTINCT order_id) AS completed_orders,
  ROUND(SUM(line_revenue), 2) AS revenue
FROM silver.order_lines
WHERE status = 'Completed'
GROUP BY channel

In [ ]:
%sql
SELECT *
FROM demo_completed_channel
ORDER BY revenue DESC

### Disposable VIEW

A view is a schema object. In this demo we create and drop it immediately to show the mechanics without leaving training clutter behind.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW v_demo_completed_channel AS
SELECT *
FROM demo_completed_channel


In [ ]:
%sql
DESCRIBE TABLE v_demo_completed_channel


In [ ]:
%sql
DROP VIEW IF EXISTS v_demo_completed_channel


### Materialized View and Streaming Table Concepts

These are platform objects for managed refresh patterns. We discuss them here, but do not execute the examples because feature availability and permissions vary by workspace.

```sql
CREATE MATERIALIZED VIEW gold.mv_monthly_revenue AS
SELECT DATE_FORMAT(order_date, 'yyyy-MM') AS year_month, SUM(line_revenue) AS revenue
FROM gold.fact_sales
WHERE is_completed
GROUP BY DATE_FORMAT(order_date, 'yyyy-MM');

CREATE OR REFRESH STREAMING TABLE silver.orders_streaming AS
SELECT * FROM STREAM read_files('/Volumes/.../orders', format => 'json');
```

Teaching point: materialized views and streaming tables are not a replacement for modeling. They are refresh and performance tools around a clear data contract.

## 5. SQL Parameters, Variables and Scripting Fundamentals

![SQL Parameters and Variables](../../../assets/images/day1_02_sql_parameters_variables.png)

How this visual helps: it separates three patterns that are often confused: parameters receive user input, variables hold session state, and scripting adds control flow.

Databricks SQL gives analysts three related patterns:

- **named parameters** for interactive SQL Editor queries and dashboards,
- **session variables** for repeatable SQL inside one session,
- **SQL scripting** for compound SQL logic, validation blocks and automation.

Hands-on step: run the session-variable cells below, then compare them with the SQL Editor parameter pattern.

Expected observation: parameters are user input; variables are state in the SQL session; scripting is a control-flow layer for repeatable SQL tasks.

### Parameters, Variables and Scripting Documentation Anchors

| Feature | Working definition | Official reference |
|---|---|---|
| Named parameter marker | runtime placeholder such as `:status`, useful for reusable SQL and dashboards | [Named parameter markers](https://docs.databricks.com/aws/en/sql/user/queries/query-parameters) |
| Parameter widget | UI control generated from a named parameter | [Parameter widgets](https://docs.databricks.com/aws/en/sql/user/sql-editor/parameter-widgets) |
| Session variable | value declared in the SQL session and reused across statements | [DECLARE VARIABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-declare-variable) |
| `SET VARIABLE` | assigns or changes a session variable value | [SET VARIABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-aux-set-variable) |
| `EXECUTE IMMEDIATE` | executes a SQL string built at runtime | [EXECUTE IMMEDIATE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-aux-execute-immediate) |
| SQL scripting | procedural SQL blocks with variables and control flow | [SQL scripting](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-scripting) |


### SQL Editor Named Parameters

In SQL Editor and Databricks SQL dashboards, parameters make a query interactive. A common pattern is to write placeholders such as `:status`, `:start_date` and `:top_n`, then let the UI provide values.

Use this when the user or dashboard should choose the value at run time. Use a session variable when the SQL session itself should carry the value across statements.

Example for SQL Editor, not executed in this notebook:

```sql
SELECT order_id, order_date, status, channel
FROM silver.sales_orders
WHERE status = :status
  AND order_date >= :start_date
ORDER BY order_date DESC
LIMIT :top_n;
```

The companion SQL file [sql/exploration_queries.sql](../../../sql/exploration_queries.sql) contains a complete SQL Editor-oriented example with named parameters.

### Session Variables: Declare Once, Reuse in SQL

Session variables apply to Databricks SQL / Databricks Runtime 14.1 and above. They are useful for repeatable notebook demos and SQL Warehouse sessions because the value lives in the current SQL session and can be reused by later statements.

In [ ]:
%sql
DECLARE OR REPLACE VARIABLE demo_status STRING DEFAULT 'Completed';

In [ ]:
%sql
DECLARE OR REPLACE VARIABLE demo_top_n INT DEFAULT 5;

In [ ]:
%sql
SELECT
  demo_status AS active_status,
  demo_top_n AS active_top_n

In [ ]:
%sql
WITH product_revenue AS (
  SELECT
    product_id,
    ROUND(SUM(line_revenue), 2) AS revenue
  FROM silver.order_lines
  WHERE status = demo_status
  GROUP BY product_id
),
ranked AS (
  SELECT
    product_id,
    revenue,
    DENSE_RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
  FROM product_revenue
)
SELECT *
FROM ranked
WHERE revenue_rank <= demo_top_n
ORDER BY revenue_rank, product_id

### Change the Variable, Not the Query

This is the analyst benefit: the query shape stays stable while the inputs change. It is easier to review and easier to compare in Query History.

### EXECUTE IMMEDIATE Awareness

`EXECUTE IMMEDIATE` runs a SQL statement assembled at runtime. It is useful for administrative or metadata-driven SQL, but it is not the default pattern for BI analysis.

Use parameter markers and variables for normal analyst filtering. Reserve dynamic SQL for cases where the object name or statement shape must be generated.

```sql
DECLARE OR REPLACE VARIABLE demo_table STRING DEFAULT 'silver.order_lines';
EXECUTE IMMEDIATE 'SELECT COUNT(*) AS rows FROM ' || demo_table;
```

Pitfall: dynamic SQL is harder to review. Do not use it to hide business filters or KPI logic.


In [ ]:
%sql
SET VARIABLE demo_status = 'Returned';

In [ ]:
%sql
SET VARIABLE demo_top_n = 3;

In [ ]:
%sql
WITH product_revenue AS (
  SELECT
    product_id,
    ROUND(SUM(line_revenue), 2) AS revenue
  FROM silver.order_lines
  WHERE status = demo_status
  GROUP BY product_id
),
ranked AS (
  SELECT
    product_id,
    revenue,
    DENSE_RANK() OVER (ORDER BY revenue DESC) AS revenue_rank
  FROM product_revenue
)
SELECT *
FROM ranked
WHERE revenue_rank <= demo_top_n
ORDER BY revenue_rank, product_id

### SQL Scripting: Orientation

![SQL scripting flow](../../../assets/images/day1_02_scripting_flow.png)

How this visual helps: it separates ordinary SQL statements from procedural SQL blocks. Participants should see scripting as a repeatable validation or automation wrapper, not as the default way to write BI queries.

SQL scripting adds compound statements such as `BEGIN ... END`, variables, conditionals and loops. In Databricks SQL it is useful when validation logic or administrative SQL should become a repeatable SQL task.

Full scripting support applies to Databricks SQL / Databricks Runtime 16.3 and above. Treat the next block as delivery material unless the classroom workspace confirms that SQL scripting is available.

```sql
BEGIN
  DECLARE completed_orders BIGINT DEFAULT 0;

  SET completed_orders = (
    SELECT COUNT(DISTINCT order_id)
    FROM silver.sales_orders
    WHERE status = 'Completed'
  );

  IF completed_orders > 0 THEN
    SELECT completed_orders AS completed_orders_checked;
  ELSE
    SELECT 'No completed orders found' AS validation_message;
  END IF;
END;
```

Professional rule: use simple SQL variables for exploration; use SQL scripting when you need a repeatable SQL-only validation or automation block.


## 6. Query Optimization Foundations for BI

This section covers the optimization concepts that analysts must recognize before Workshop 1 and Power BI modeling.

Definitions:

- `EXPLAIN` shows the query plan without running the query.
- `EXPLAIN FORMATTED` gives a more readable physical plan with scan, filter and aggregation details.
- **Partition pruning** means Databricks can skip partitions that cannot match a filter.
- **Scan reduction** means reading fewer columns or fewer files before aggregation.
- **Pre-aggregation** means answering a BI page from a smaller grouped dataset instead of scanning detail rows every time.

Expected observation: better BI SQL is not only shorter. It has explicit filters, narrow projections and a clear grain.

### Optimization Documentation Anchors

| Feature | Working definition | Official reference |
|---|---|---|
| `EXPLAIN` | shows logical or physical query plans without returning the query result | [EXPLAIN](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-qry-explain) |
| Query Profile | runtime evidence for scans, joins, spills and expensive operators | [SQL warehouse monitoring](https://docs.databricks.com/aws/en/compute/sql-warehouse/monitor) |
| Liquid clustering | data layout optimization that can improve filter and join performance | [Liquid clustering](https://docs.databricks.com/aws/en/tables/clustering) |
| `OPTIMIZE` | compacts or reorganizes table data to improve read efficiency | [SQL language reference](https://docs.databricks.com/aws/en/sql/language-manual/) |


### EXPLAIN: Broad Detail Query

This query answers a real BI question, but it scans line-grain data and joins dimensions. Use the plan to identify scans, joins and aggregations.

In [ ]:
%sql
EXPLAIN FORMATTED
SELECT
  p.category,
  c.region,
  ROUND(SUM(ol.quantity * ol.unit_price), 2) AS revenue
FROM silver.order_lines ol
JOIN silver.sales_orders so ON ol.order_id = so.order_id
JOIN silver.products p ON ol.product_id = p.product_id
JOIN silver.customers c ON so.customer_id = c.customer_id
WHERE so.order_date >= DATE '2024-01-01'
  AND so.order_date < DATE '2025-01-01'
  AND so.status = 'Completed'
GROUP BY p.category, c.region
ORDER BY revenue DESC

Expected observation: the plan should reveal detail-table scans, joins and a final aggregate. This is acceptable for modeling, but risky as a repeated DirectQuery visual pattern.

### Scan Reduction: Filter and Project Early

A BI query should select only the columns needed for the visual and apply high-selectivity filters before aggregation. CTEs make this reviewable.

In [ ]:
%sql
WITH filtered_orders AS (
  SELECT order_id, customer_id
  FROM silver.sales_orders
  WHERE order_date >= DATE '2024-01-01'
    AND order_date < DATE '2025-01-01'
    AND status = 'Completed'
),
filtered_lines AS (
  SELECT order_id, product_id, quantity, unit_price
  FROM silver.order_lines
)
SELECT
  p.category,
  c.region,
  ROUND(SUM(fl.quantity * fl.unit_price), 2) AS revenue
FROM filtered_lines fl
JOIN filtered_orders fo ON fl.order_id = fo.order_id
JOIN silver.products p ON fl.product_id = p.product_id
JOIN silver.customers c ON fo.customer_id = c.customer_id
GROUP BY p.category, c.region
ORDER BY revenue DESC

Expected observation: the result matches the broad query, but the structure makes filter placement, selected columns and aggregation grain easier to review.

### Pre-aggregation Pattern

Pre-aggregation is a common BI serving pattern: calculate a smaller grouped result once, then let dashboards query that smaller shape.

In this demo we use a `TEMP VIEW`; Workshop 2 will build durable serving objects for Power BI.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW demo_bi_monthly_revenue AS
SELECT
  date_format(so.order_date, 'yyyy-MM') AS year_month,
  p.category,
  c.region,
  ROUND(SUM(ol.quantity * ol.unit_price), 2) AS revenue
FROM silver.order_lines ol
JOIN silver.sales_orders so ON ol.order_id = so.order_id
JOIN silver.products p ON ol.product_id = p.product_id
JOIN silver.customers c ON so.customer_id = c.customer_id
WHERE so.status = 'Completed'
GROUP BY date_format(so.order_date, 'yyyy-MM'), p.category, c.region

In [ ]:
%sql
SELECT
  year_month,
  category,
  region,
  revenue
FROM demo_bi_monthly_revenue
WHERE year_month >= '2024-01'
ORDER BY year_month, revenue DESC

Expected observation: dashboards should prefer this smaller monthly shape when the visual does not need line-level detail.

## 7. Medallion Architecture

![Medallion to Power BI](../../../assets/images/medallion_to_powerbi.png)

How this visual helps: it gives participants the end-to-end mental model for the course: raw and cleaned data become governed Gold objects, and Power BI consumes the Gold contract instead of reconnecting to every source table.

Medallion is a quality and consumption architecture for the lakehouse.

| Layer | Responsibility | BI meaning |
|---|---|---|
| Bronze | raw landing and traceability | not a report source |
| Silver | cleaned and conformed business entities | reusable source for modeling |
| Gold | curated facts, dimensions, aggregates and views | Power BI contract |

Gold should expose a clear grain, approved dimensions, shared KPI logic and refresh expectations.

### Notebook Modularity Pattern

Each Medallion layer is implemented as an **independent notebook** that reads one table and writes one table. The `dbutils.notebook.run()` API chains them from an orchestrator notebook.

```python
# Orchestrator: run Bronze → Silver → Gold in sequence
dbutils.notebook.run(
    "materials/medallion/bronze_customers",
    timeout_seconds=300,
    arguments={"catalog": CATALOG, "schema": BRONZE_SCHEMA, "source_path": DATASET_PATH}
)
dbutils.notebook.run(
    "materials/medallion/silver_customers",
    timeout_seconds=300,
    arguments={"catalog": CATALOG, "schema_bronze": BRONZE_SCHEMA, "schema_silver": SILVER_SCHEMA}
)
dbutils.notebook.run(
    "materials/medallion/gold_customer_orders_summary",
    timeout_seconds=300,
    arguments={"catalog": CATALOG, "schema_silver": SILVER_SCHEMA, "schema_gold": GOLD_SCHEMA}
)
```

**Rules that make this pattern work:**

| Rule | Implementation |
|---|---|
| One input → one output | Each notebook reads one table, writes one table |
| Arguments via widgets | `dbutils.widgets.get("catalog")` — testable in isolation |
| Structured return | `dbutils.notebook.exit(json.dumps({"status": "ok", "rows": n}))` |
| No hardcoded paths | Catalog, schema, source path all passed as arguments |
| Idempotent | Run multiple times = same result (`overwrite` by design) |

**Day 2 connection:** In Lakeflow Jobs, each `dbutils.notebook.run()` call becomes a **Notebook Task** in a DAG. Databricks handles retry logic, alerting and scheduling automatically — the same modular notebooks, now production-grade.

## 8. Analytics Architecture Patterns

![Architecture patterns deep dive](../../../assets/images/day1_02_architecture_patterns_deep_dive.png)

How this visual helps: it shows how architecture patterns can coexist by layer and purpose. The lakehouse provides the Bronze/Silver/Gold structure, while Kimball, Inmon, Data Vault and Data Mesh answer different modeling and ownership questions.

![Architecture patterns to Power BI Gold](../../../assets/images/architecture_patterns_to_powerbi.png)

How this visual helps: it links architecture vocabulary to the Power BI outcome. The point is not to memorize patterns, but to decide which pattern shapes the Gold contract.

These patterns answer different questions. They are not interchangeable buzzwords.

| Pattern | Main idea | Power BI takeaway |
|---|---|---|
| Kimball | facts and dimensions | best default for BI semantic models |
| Inmon | integrated enterprise warehouse core | often upstream from marts |
| Data Vault | historized hubs, links, satellites | good for audit and many sources; publish marts on top |
| Data Mesh | domain-owned data products | clarifies ownership of Gold contracts |
| Medallion | quality and consumption layers | Databricks lakehouse layering pattern |

### Kimball - dimensional model for analytics

![Kimball dimensional model](../../../assets/images/kimball_dimensional_model_ai.png)

How this visual helps: it makes the star-schema shape visible. Facts hold measurable events, dimensions hold slicer context, and Power BI relationships become easier to reason about.

Kimball starts from the reporting questions users ask and models data as **facts** and **dimensions**. A fact table stores measurable business events at a clear grain, for example one sales order line. Dimension tables describe the context of those events, for example customer, product, date or store.

This is why Kimball is usually the easiest shape for Power BI and other BI tools: measures live in facts, slicers and labels live in dimensions, and joins are predictable. In a Databricks lakehouse, Kimball most often appears in the **Gold** layer as a star schema or a small set of related stars.

Use Kimball when the main goal is fast, understandable analytics: dashboards, semantic models, self-service BI and repeatable KPIs. The trade-off is that dimensional models are optimized for consumption, not for preserving every source-system detail or every historical integration decision.

### Inmon - integrated enterprise warehouse core

![Inmon integrated warehouse core](../../../assets/images/inmon_integrated_warehouse_core_ai.png)

How this visual helps: it contrasts integration-first architecture with BI-first dimensional modeling. Participants should see why an enterprise core often feeds marts instead of being the direct report model.

Inmon starts from the enterprise integration problem. The central warehouse is modelled top-down, commonly in a normalized form, so core business entities such as customers, products, contracts and transactions have consistent definitions across the organization.

This gives strong governance and a single integrated version of business data before it is published into downstream marts. The cost is read-time complexity: normalized models are clean for integration, but usually require more joins and are less convenient for direct BI consumption than a Kimball star.

In a Databricks lakehouse, Inmon maps naturally to a curated **Silver** integration layer, with BI-facing Kimball marts published in **Gold**. In practice, Inmon and Kimball are not enemies: many mature platforms use Inmon-style integration upstream and Kimball-style dimensional models downstream.

For this course: Medallion explains the layers, Kimball shapes the BI-facing Gold model, and Data Mesh language helps talk about ownership. Data Vault and Inmon are usually upstream context, not report-layer replacements.


## 9. SCD Type 1 vs Type 2

Slowly changing dimensions decide what historical reports show when an attribute changes. Workshop 1 builds a clean model; here we only compare the behavior with small disposable examples.

### SCD and CDC Documentation Anchors

| Concept | Working definition | Official reference |
|---|---|---|
| SCD Type 1 | overwrite attribute values; reports show the current value only | [AUTO CDC examples](https://docs.databricks.com/aws/en/ldp/cdc) |
| SCD Type 2 | preserve effective-dated historical versions | [AUTO CDC examples](https://docs.databricks.com/aws/en/ldp/cdc) |
| `MERGE INTO` | manual Delta DML pattern for upserts and conditional updates | [MERGE INTO](https://docs.databricks.com/aws/en/sql/language-manual/delta-merge-into) |
| AUTO CDC / `APPLY CHANGES INTO` | pipeline feature that handles ordered CDC changes and supports SCD Type 1/2 | [AUTO CDC INTO](https://docs.databricks.com/aws/en/ldp/developer/ldp-sql-ref-apply-changes-into) |
| `dlt.apply_changes` / `create_auto_cdc_flow` | Python API for declarative CDC processing in Lakeflow pipelines | [AUTO CDC Python API](https://docs.databricks.com/aws/en/ldp/developer/ldp-python-ref-apply-changes) |


### SCD Type 1: Overwrite Current Value

Type 1 keeps only the current attribute value. It is simple and often correct for corrections, but it rewrites history for reporting.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW demo_scd_type1 AS
SELECT 101 AS customer_id, 'Acme Corp' AS customer_name, 'Enterprise' AS segment
UNION ALL
SELECT 102, 'Global Inc', 'Enterprise'

In [ ]:
%sql
SELECT *
FROM demo_scd_type1

### SCD Type 2: Preserve History

Type 2 stores effective-dated versions. Reports can ask what the customer attribute was at the time of the event.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW demo_scd_type2 AS
SELECT 101 AS customer_id, 'Acme Corp' AS customer_name, 'SMB' AS segment,
       DATE '2024-01-01' AS valid_from, DATE '2025-05-14' AS valid_to, false AS is_current
UNION ALL
SELECT 101, 'Acme Corp', 'Enterprise', DATE '2025-05-15', DATE '9999-12-31', true
UNION ALL
SELECT 102, 'Global Inc', 'Enterprise', DATE '2024-01-01', DATE '9999-12-31', true

In [ ]:
%sql
SELECT customer_id, customer_name, segment
FROM demo_scd_type2
WHERE customer_id = 101
  AND DATE '2025-01-15' BETWEEN valid_from AND valid_to

### AUTO CDC — `APPLY CHANGES INTO` (Lakeflow Pipelines)

Manual `MERGE INTO` for SCD Type 2 requires careful orchestration. **Lakeflow Pipelines** (DLT) provides `APPLY CHANGES INTO` — a declarative API that handles CDC (Change Data Capture) automatically.

```python
import dlt
from pyspark.sql.functions import col

# Step 1: Define the Bronze streaming table (source)
@dlt.table(name="bronze_customers")
def bronze_customers():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .load("/Volumes/retail/landing/customers/")
    )

# Step 2: Apply CDC — SCD Type 1 (overwrite on change)
dlt.apply_changes(
    target      = "silver_customers",    # destination table
    source      = "bronze_customers",    # streaming source
    keys        = ["customer_id"],       # business key
    sequence_by = col("_load_ts"),       # latest version wins
    stored_as_scd_type = 1               # 1 = overwrite | 2 = add row with is_current flag
)
```

**Manual MERGE vs. AUTO CDC:**

| Manual (`MERGE INTO`) | AUTO CDC (`dlt.apply_changes`) |
|---|---|
| `WHEN MATCHED ... UPDATE SET` | `sequence_by = col("_load_ts")` handles ordering automatically |
| `WHEN NOT MATCHED ... INSERT` | Handled by pipeline engine |
| SCD Type 2 requires explicit `is_current`, `valid_from`, `valid_to` logic | `stored_as_scd_type = 2` adds these columns automatically |
| Runs as notebook job | Runs inside a Lakeflow Pipeline (DLT context) |

> **Note:** `APPLY CHANGES INTO` is available only inside a Lakeflow Pipeline (`.py` file with `import dlt`). Full hands-on coverage is in **Day 3 — Lakeflow Pipelines**.

## 10. Metric View and Semantic Layer Awareness

![KPI definition flow](../../../assets/images/kpi_definition_flow.png)

How this visual helps: it shows why KPI ownership must be explicit. A business question becomes a KPI definition, SQL logic, a Gold object and finally a dashboard; duplicating that logic in many reports creates inconsistent numbers.

A metric view is a governed semantic object that defines measures and dimensions closer to the data platform. It can reduce duplicated KPI logic across BI tools when the organization wants central metric definitions.

A SQL view defines a reusable query result. A metric view defines business metrics and dimensional context. A Power BI semantic model adds relationships, DAX measures, hierarchies, formatting and report-specific behavior.

Practical rule for this course: build trusted Gold objects in Workshop 1, then decide in Day1_03 which measures belong in Power BI and which should be governed closer to Databricks.

Documentation: [Unity Catalog metric views](https://docs.databricks.com/aws/en/business-semantics/metric-views/), [create and edit metric views](https://docs.databricks.com/aws/en/business-semantics/metric-views/create-edit), [metric view YAML reference](https://docs.databricks.com/aws/en/business-semantics/metric-views/yaml-reference), [BI compatibility mode](https://docs.databricks.com/aws/en/partners/bi/bi-metric-view).

Expected observation: semantic-layer choices are ownership choices. They decide where KPI definitions live and who can change them.


## Summary and Workshop Handoff

This notebook showed SQL and architecture patterns in small examples. It deliberately did not build the final Gold model.

Workshop 1 now uses these patterns to create a full Kimball model for Power BI:

- source grain checks become fact and dimension grain decisions,
- CTEs become readable model-building SQL,
- variables and parameters become repeatable validation patterns,
- SCD choices become explicit dimension behavior,
- Gold objects become the Power BI contract.